# Train DermaLens in the cloud (Google Colab)

This notebook trains the DermaLens model on a free cloud GPU, so the ~3 GB dataset never touches your own computer.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. It takes roughly 30–45 minutes (training stops early once it stops improving).

At the end your browser downloads three files:

- `dermalens_efficientnet_b0.pt` — the trained model (~20 MB)
- `evaluation.json` — held-out test results
- `training_history.json` — loss/accuracy per epoch

Upload the first two to a GitHub Release so the online API can load them (see `DEPLOY.md`).

## Dataset licence — read this

HAM10000 is published on the Harvard Dataverse (doi:10.7910/DVN/DBW86T) under **CC BY-NC 4.0**: non-commercial use only, with attribution. Running the download cell means you accept those terms. Cite it as:

> Tschandl P., Rosendahl C. & Kittler H. *The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions.* Sci. Data 5, 180161 (2018).

In [ ]:
# 1. Check the GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU: set Runtime → Change runtime type → T4 GPU, then rerun.")

In [ ]:
# 2. Get the DermaLens code
REPO = "https://github.com/SahilGBhuva/DermaLens.git"
BRANCH = "redesign-3d-and-ml"
!rm -rf DermaLens && git clone --depth 1 --branch {BRANCH} {REPO}
%cd DermaLens
!pip install -q pandas scikit-learn pillow

In [ ]:
# 3. Download HAM10000 from the Harvard Dataverse (~2.8 GB, stays in Colab)
import pathlib, subprocess
raw = pathlib.Path("data/raw"); (raw / "images").mkdir(parents=True, exist_ok=True)
base = "https://dataverse.harvard.edu/api/access/datafile"
files = {
    "HAM10000_metadata.csv": f"{base}/4338392?format=original",
    "part_1.zip": f"{base}/3172585",
    "part_2.zip": f"{base}/3172584",
}
for name, url in files.items():
    print("downloading", name)
    subprocess.run(["wget", "-q", "-O", str(raw / name), url], check=True)
for part in ("part_1.zip", "part_2.zip"):
    subprocess.run(["unzip", "-q", "-o", str(raw / part), "-d", str(raw / "images")], check=True)
    (raw / part).unlink()
print(len(list((raw / "images").glob("*.jpg"))), "images")

In [ ]:
# 4. Split by lesion, check for leakage, train, then evaluate once on the test split
!python ml/run_pipeline.py \
  --metadata data/raw/HAM10000_metadata.csv \
  --images-dir data/raw/images \
  --epochs 25 \
  --batch-size 64

In [ ]:
# 5. Headline results
import json
m = json.load(open("models/evaluation.json"))
for key in ["accuracy", "balanced_accuracy", "macro_f1", "macro_ovr_roc_auc", "expected_calibration_error"]:
    value = m.get(key)
    print(f"{key:28s}", "—" if value is None else f"{value:.3f}")
print()
for label, r in m["per_class_sensitivity_specificity"].items():
    print(f"{label:6s} sensitivity {r['sensitivity']:.2f}   specificity {r['specificity']:.2f}")

In [ ]:
# 6. Download the results to your computer
import hashlib
from google.colab import files
for name in ["dermalens_efficientnet_b0.pt", "evaluation.json", "training_history.json"]:
    path = f"models/{name}"
    print(name, "sha256:", hashlib.sha256(open(path, "rb").read()).hexdigest())
    files.download(path)